# Aivora AI - LoRA fine-tuning of an open pretrained model

The from-scratch 101M model has plateaued: its last pretraining session gained
0.04 val loss over 46,000 steps, instruction tuning overfitted at two
different learning rates, and its evaluation score (1-5 of 45) never left its
own noise band.

This notebook takes **Qwen2.5-1.5B-Instruct** (Apache-2.0, pretrained on
trillions of tokens) and adapts it to the same finance data with LoRA - only
~1% of parameters are trained, so it fits the same free T4 and produces a
~50 MB adapter instead of a 3 GB model.

The comparison is kept honest:

* the **same 45 evaluation questions** and the same scorer as the from-scratch
  model (`evaluation/generic.py`);
* the base model is scored **before** tuning, in the same session;
* greedy decoding (no sampling), because sampling made the project's own
  scores swing 1-5 of 45 on identical weights.


## 1. Environment

In [ ]:
import platform, sys, os
print("Python:", sys.version)
print("Platform:", platform.platform())
print("CWD:", os.getcwd())
print("Kaggle input mounted:", os.path.exists("/kaggle/input"), os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else [])


## 1b. GPU compute-capability check (before the first `import torch`)

Kaggle has been assigning this account a Tesla P100 (compute capability
6.0 / sm_60) rather than a T4, and the base image's shipped torch build
only supports compute capability 7.0+ - every real CUDA kernel launch
fails with `AcceleratorError: no kernel image is available` regardless of
`batch_size` unless an older, wider-compatibility torch build is used.

Checked via `nvidia-smi` (not `torch.cuda.get_device_capability()`)
specifically so this runs **before** `import torch` - reinstalling torch
mid-process and `importlib.reload()`-ing it is not safe (torch's C
extension re-registers native `TORCH_LIBRARY` namespaces with the
dispatcher, which crashes on a second registration). If the attached GPU
needs an older, wider-compatibility torch build, it's installed here,
before torch is ever imported for the first time.

In [ ]:
import subprocess, sys

# Install peft only. Upgrading transformers here broke the first attempt:
# transformers 5.17 requires torchao > 0.16 while this image ships 0.10, so
# importing Trainer raised ImportError after the base model had already been
# loaded and scored. The image's own transformers matches its own torchao.
# --no-deps matters: a plain install upgraded huggingface_hub, which broke
# this image's transformers 5.0.0 with StrictDataclassDefinitionError while
# loading the Qwen config. peft's dependencies are all present already.
import huggingface_hub
print('huggingface_hub before:', huggingface_hub.__version__)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'peft'],
                   capture_output=True, text=True)
print(r.stdout[-800:]); print(r.stderr[-800:])
if r.returncode != 0:
    raise RuntimeError('STATUS = BLOCKED: pip install peft failed (see above).')

import importlib
importlib.reload(huggingface_hub)
print('huggingface_hub after :', huggingface_hub.__version__)
import accelerate, peft, torch, transformers
print(f'transformers {transformers.__version__} | peft {peft.__version__} | '
      f'accelerate {accelerate.__version__} | torch {torch.__version__}')
try:
    import torchao
    print('torchao', torchao.__version__)
except Exception as e:
    print('torchao not importable:', e)

# Qwen2.5 needs transformers >= 4.37.
major, minor = (int(x) for x in transformers.__version__.split('.')[:2])
if (major, minor) < (4, 37):
    raise RuntimeError(f'STATUS = BLOCKED: transformers {transformers.__version__} is too old for Qwen2.5.')

from transformers import Trainer  # fail here, not after a 10-minute model load
print('Trainer imports cleanly')

## 2. GPU / CUDA verification (hard gate)

Raises immediately if no GPU is attached - never claims GPU training happened without this passing.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "STATUS = BLOCKED: torch.cuda.is_available() is False. "
        "Go to Settings (right sidebar) -> Accelerator -> GPU T4 x2, "
        "save, and re-run this notebook from the top."
    )

GPU_NAME = torch.cuda.get_device_name(0)
GPU_COUNT = torch.cuda.device_count()
CC = torch.cuda.get_device_capability(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)

print("GPU AVAILABLE =", True)
print("GPU name:", GPU_NAME)
print("GPU count:", GPU_COUNT)
print("Compute capability:", CC)
print("Total VRAM (GPU 0): %.2f GB" % total_vram_gb)
print("torch version:", torch.__version__, "| CUDA build:", torch.version.cuda)


## 3. Repository transfer + integrity check

Clones the real, public repo. If this fails, Internet is probably off (Settings -> Internet -> On).

In [ ]:
import subprocess, os

REPO_URL = "https://github.com/Ankushk-aosc/Aivora-AI.git"
REPO_DIR = "/kaggle/working/Aivora-AI"

if not os.path.exists(REPO_DIR):
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
                             capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            "STATUS = BLOCKED: git clone failed (see stderr above). "
            "Most likely cause: Internet is off for this notebook "
            "(Settings -> Internet -> On), or the repo URL changed."
        )
else:
    print(f"{REPO_DIR} already present, skipping clone.")

os.chdir(REPO_DIR)
print("Now in:", os.getcwd())

required_paths = [
    "models/model.py", "training/trainer.py", "ai_platform/model_registry.py",
    "data_sources/prepare.py", "evaluation/evaluator.py", "configs/small.yaml",
    "configs/financial_poc.yaml", "inference/generator.py",
]
missing = [p for p in required_paths if not os.path.exists(p)]
if missing:
    raise RuntimeError(f"STATUS = BLOCKED: repo clone incomplete, missing {missing}")
print("Repo integrity check passed:", len(required_paths), "required paths present.")


## 4. Dependencies

Installed here rather than reusing the pretraining notebook's cell: this run
needs `peft` for LoRA, and a `transformers` new enough for the Trainer
arguments used below.

In [ ]:
import subprocess, sys

r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                    'transformers', 'peft', 'accelerate'],
                   capture_output=True, text=True)
print(r.stdout[-1500:]); print(r.stderr[-1500:])
if r.returncode != 0:
    raise RuntimeError('STATUS = BLOCKED: dependency install failed (see above).')

import transformers, peft, accelerate, torch
print(f'transformers {transformers.__version__} | peft {peft.__version__} | '
      f'accelerate {accelerate.__version__} | torch {torch.__version__}')

## 5. One GPU, not two

`Trainer` would wrap the model in DataParallel across Kaggle's T4 x2, which
splits the batch and complicates LoRA saving for no throughput gain at this
size. One T4 (16 GB) fits a 1.5B model in fp16 with LoRA comfortably.

In [ ]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import torch
print('visible GPUs:', torch.cuda.device_count())
print('GPU:', torch.cuda.get_device_name(0))
print('memory: %.1f GiB' % (torch.cuda.get_device_properties(0).total_memory / 1024**3))

## 6. Build the finance instruction dataset

The same builder and the same leakage check the from-scratch run used, so both
models are trained on comparable data and neither sees an evaluation
question.

In [ ]:
from data_sources.build_instruction_dataset import build

stats = build()
print('')
print(f"examples: {stats['total']:,} | by source: {stats['by_source']}")
print(f"dropped:  {stats['dropped']}")

import json
records = [json.loads(line) for line in open(stats['path'], encoding='utf-8')]
import random
random.Random(42).shuffle(records)

# ~12k examples is about one hour of T4 time at seq_len 1024; the rest is
# held out rather than rushed through.
TRAIN_N, VAL_N = 12000, 400
train_records = records[:TRAIN_N]
val_records = records[TRAIN_N:TRAIN_N + VAL_N]
print(f'train {len(train_records):,} | val {len(val_records):,}')

## 7. Load the base model and score it BEFORE tuning

This is the number the tuned model has to beat. Same questions, same scorer,
greedy decoding.

In [ ]:
from evaluation import print_report
from evaluation.generic import evaluate_generator
from training.hf_lora_sft import DEFAULT_MODEL, load_base_model, make_answerer

print(f'Loading {DEFAULT_MODEL} ...')
model, tokenizer = load_base_model(DEFAULT_MODEL)
print('parameters: %.2fB' % (sum(p.numel() for p in model.parameters()) / 1e9))

base_results = evaluate_generator(make_answerer(model, tokenizer), verbose=True,
                                  label='Qwen2.5-1.5B-Instruct (BEFORE tuning)')
print_report(base_results)
BASE_SCORE = base_results['overall']['accuracy']

## 8. Fine-tune with LoRA

Loss is taken on the answer tokens only (the prompt is masked), so the model
learns to answer rather than to echo the question.

In [ ]:
from training.hf_lora_sft import attach_lora, tokenize_records, train_lora

train_rows = tokenize_records(tokenizer, train_records, max_len=1024)
val_rows = tokenize_records(tokenizer, val_records, max_len=1024)
print(f'tokenized: train {len(train_rows):,} | val {len(val_rows):,}')

model = attach_lora(model, r=16, alpha=32, dropout=0.05)
model.config.use_cache = False  # required with gradient checkpointing

OUT_DIR = '/kaggle/working/qwen_finance_lora'
trainer, result, metrics = train_lora(
    model, tokenizer, train_rows, val_rows, OUT_DIR,
    epochs=1, batch_size=2, grad_accum=8, learning_rate=2e-4,
    eval_steps=100, max_train_seconds=4 * 3600,
)
print('')
print(f"train loss {result.training_loss:.4f} | eval loss {metrics['eval_loss']:.4f}")

## 9. Score the tuned model and compare

In [ ]:
model.config.use_cache = True
model.eval()
tuned_results = evaluate_generator(make_answerer(model, tokenizer), verbose=True,
                                   label='Qwen2.5-1.5B + finance LoRA (AFTER tuning)')
print_report(tuned_results)
TUNED_SCORE = tuned_results['overall']['accuracy']
print('')
print(f'Qwen base           : {BASE_SCORE}%')
print(f'Qwen + finance LoRA : {TUNED_SCORE}%')
print('from-scratch 101M model (for reference): 2.22-11.11% depending on sampling')

## 10. Sample answers

Read these rather than trusting the score alone - the scorer only checks for
keywords and numbers.

In [ ]:
answer = make_answerer(model, tokenizer, max_new_tokens=128)
for q in ['What is EBITDA?',
          'What is working capital?',
          'Explain gross margin in one sentence.',
          'What does a balance sheet show?',
          'Why do companies issue bonds?',
          'Calculate EBITDA margin for revenue 500 and EBITDA 100.']:
    print(f'\nQ: {q}\nA: {answer(q)}')

## 11. Save the adapter

The adapter is small (~50 MB) because the 3 GB base model is unchanged - it is
downloaded from Hugging Face at load time and the adapter applied on top.

In [ ]:
import json, os

sizes = {f: os.path.getsize(os.path.join(OUT_DIR, f))
         for f in os.listdir(OUT_DIR) if os.path.isfile(os.path.join(OUT_DIR, f))}
print('adapter files:', {k: f'{v/1e6:.1f} MB' for k, v in sizes.items()})

manifest = {
    'base_model': DEFAULT_MODEL,
    'adapter_dir': OUT_DIR,
    'train_examples': len(train_rows),
    'val_examples': len(val_rows),
    'train_loss': result.training_loss,
    'eval_loss': metrics['eval_loss'],
    'score_before': BASE_SCORE,
    'score_after': TUNED_SCORE,
    'instruction_data': stats,
}
with open('/kaggle/working/lora_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2, default=str)
print('')
print(json.dumps({k: v for k, v in manifest.items() if k != 'instruction_data'},
                 indent=2, default=str))